# Building Multi-Agent Systems with CrewAI

A minimal, working CrewAI setup that runs on **Groq** (free, fast inference).

This notebook builds a three-agent _crew_ — a **Senior Software Engineer** that
researches key points on a topic, a **Senior AI Engineer** that turns them into an
architecture plan, and a **Level 2 SOC Analyst** that writes a short security brief.

**How to run:** use _Run All_ (top toolbar) so every cell executes top-to-bottom
in order. Running a single cell in isolation will fail because later cells depend
on variables defined earlier.

## 1. Install dependencies

`crewai` pulls in `litellm` (the layer that talks to Groq) and everything else.
`python-dotenv` loads your API key from a local `.env` file.


In [1]:
%pip install -q crewai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load the API key and configure the LLM

Create a file named `.env` next to this notebook containing:

```
GROQ_API_KEY=your_real_key_here
```

Get a free key at https://console.groq.com/keys. The `.env` file is git-ignored,
so your key never gets committed.


In [2]:
from dotenv import load_dotenv
import os
from crewai import LLM

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY not found. Create a .env file next to this notebook "
        "with a line like:  GROQ_API_KEY=your_key_here"
    )

# Groq models are addressed as "groq/<model-name>" via LiteLLM.
llm = LLM(model="groq/llama-3.1-8b-instant", api_key=api_key)

print("LLM configured:", llm.model)

LLM configured: groq/llama-3.1-8b-instant


## 3. Compatibility shim for Groq

CrewAI 1.15.x tags every message with a `cache_breakpoint` field — a prompt-caching
hint that only Anthropic's API understands. When you route through Groq, that extra
field is sent verbatim and Groq's strict schema rejects it:

> `property 'cache_breakpoint' is unsupported`

The line below neutralizes the tagging so the same code works on any provider.
Remove it once you switch to Anthropic or once CrewAI fixes this upstream.


In [3]:
import crewai.llms.cache as _crewai_cache

# Make the cache-breakpoint tag a no-op so non-Anthropic providers accept the request.
_crewai_cache.mark_cache_breakpoint = lambda message: message

print("Groq compatibility shim applied.")

Groq compatibility shim applied.


## 4. Define the agents

Each agent has a **role**, a **goal**, and a **backstory** that shapes how it behaves.
Both agents share the same `llm` we configured above.


In [8]:
from crewai import Agent

software_engineer = Agent(
    role="Senior Software Engineer",
    goal="Find and summarize the most important, current facts about, and ways on how to build worldclass ai agents in 2026 {topic}",
    backstory=(
        "You are a topclass experienced senior Software engineer"
        "You are a master in onboarding teams, from scratch, you know how to intergrate AI models into software and build the worlds best AI agents."
    ),
    llm=llm,
    verbose=True,
)

Senior_AI_engineer = Agent(
    role="Senior AI engineer",
    goal="Turn best coding practices on {topic} into a clear, engaging and intergratable ai agents that even interns can code and develop, but secure ai agents are important.",
    backstory=(
        "You are an exeperienced  ai engineer, that is well vest in machine learning, datascience, python, secure coding  and business analysis  turning business processes into real world bussiness logic. "
        "You are a team player, can lead."
        "You can develop secure ai agents."
    ),
    llm=llm,
    verbose=True,
)

Level_2_SOC_Analyst = Agent(
    role="Level 2 SOC Analyst",
    goal="Pentest and generate resports for the 2 engineers {topic} into a clear, engaging short brief",
    backstory=(
        "You are an experienced pentester, cyber practitioner, best cyber practioner in Namibia "
        "reader can absorb in under a minute."
    ),
    llm=llm,
    verbose=True,
)

## 5. Define the tasks

Tasks describe _what_ to do and the _expected output_. The `{topic}` placeholder is
filled in at run time from the `inputs` dict. `research_task` is defined first because
the architecture and security tasks reference it in their `context`, so the
researcher's output is passed to those agents automatically.

In [ ]:
from crewai import Task

# Define research_task FIRST, because the later tasks reference it in `context`.
research_task = Task(
    description="Research {topic} and list the key points a newcomer should know.",
    expected_output="4-6 concise bullet points, each one fact or insight.",
    agent=software_engineer,
)

architecture_task = Task(
    description="Outline the best architecture diagram for this project on {topic}.",
    expected_output="A clear architecture diagram, easy to understand and to implement.",
    agent=Senior_AI_engineer,
    context=[research_task],
)

security_task = Task(
    description="Recommend the safest security measures the team should consider for this project on {topic}.",
    expected_output="A 2-3 paragraph brief in plain, engaging language.",
    agent=Level_2_SOC_Analyst,
    context=[research_task],
)

## 6. Assemble and run the crew

`Process.sequential` runs tasks in order: research first, then writing.

Jupyter already runs an asyncio event loop, so we use `await crew.kickoff_async(...)`
instead of the synchronous `crew.kickoff(...)` (which raises inside a running loop).


In [ ]:
from crewai import Crew, Process

crew = Crew(
    agents=[software_engineer, Senior_AI_engineer, Level_2_SOC_Analyst],
    tasks=[research_task, architecture_task, security_task],
    process=Process.sequential,
    verbose=True,
)

result = await crew.kickoff_async(inputs={"topic": "multi-agent AI systems"})

print()
print("=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)

## 7. Inspect the run

`result` is a `CrewOutput` object. You can pull out each task's individual output
and the token usage for the whole run.


In [7]:
print("Token usage:", result.token_usage)
print()
for i, task_output in enumerate(result.tasks_output, start=1):
    print(f"--- Task {i}: {task_output.agent} ---")
    print(task_output.raw)
    print()

Token usage: total_tokens=3556 prompt_tokens=1790 cached_prompt_tokens=0 completion_tokens=1766 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=4

--- Task 1: Research Analyst ---
**Key Points for a Newcomer to Multi-Agent AI Systems**

• **Definition and Scope:** Multi-agent AI systems involve a collective of autonomous and interacting agents that collaborate to achieve a shared goal or objective through a decentralized decision-making process. These agents can be software components, robotic agents, or entities in a virtual environment. The scope of multi-agent systems includes applications in areas such as robotics, autonomous vehicles, smart grids, social networks, and economics.

• **Agent Types and Relationships:** Agents in a multi-agent system can be categorized into different types, including: (1) simple agents with independent decision-making capabilities, (2) complex agents with internal knowledge and decision-making processes, and (3) hybrid agents that combi